# Projet MLA — Prédiction de Réussite des Étudiants
## Méthodologie CRISP-DM

**Groupe :** [Votre groupe]  
**Année académique :** 2025-2026  
**Dataset :** [UCI Student Performance Dataset](https://archive.ics.uci.edu/dataset/320/student+performance)

---

## 1. Compréhension Métier (Business Understanding)

### Thématique
Analyse des facteurs influençant la réussite scolaire et prédiction du résultat des examens de rattrapage.

### ODD — Objectif de Développement Durable
**SDG 4 — Éducation de Qualité** : Assurer l'accès de tous à une éducation de qualité, sur un pied d'égalité, et promouvoir les possibilités d'apprentissage tout au long de la vie.

Ce projet contribue à l'ODD 4 en :
- Identifiant tôt les étudiants à risque d'échec pour un soutien ciblé
- Réduisant les inégalités académiques grâce à une intervention proactive
- Optimisant les ressources pédagogiques vers les profils qui en ont le plus besoin

### Tableau Récapitulatif BO / DSO / Algorithmes

| **BO (Business Objective)** | **DSO (Data Science Objective)** | **Algorithme** | **Évaluation** |
|---|---|---|---|
| Identifier les étudiants à risque d'échec | Réduire la dimensionnalité des features étudiants | ACP (PCA) | Variance expliquée cumulée ≥ 95% |
| Segmenter les étudiants par profil académique | Regrouper les étudiants en clusters homogènes | K-Means (k=3) | Silhouette Score, Méthode Elbow |
| Prédire la réussite à l'examen de rattrapage | Classifier réussite/échec (G3 ≥ 10) | Random Forest | Accuracy, F1-Score, ROC-AUC |

---
## 2. Compréhension des Données (Data Understanding)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid')

# Chargement du dataset
# Télécharger depuis : https://archive.ics.uci.edu/dataset/320/student+performance
# Placer student-mat.csv dans le même dossier que ce notebook
try:
    df = pd.read_csv('student-mat.csv', sep=';')
    print(f'Dataset chargé : {df.shape[0]} étudiants, {df.shape[1]} features')
except FileNotFoundError:
    print('Dataset non trouvé. Génération de données synthétiques...')
    import sys; sys.path.insert(0, '..')
    from app.train import generate_synthetic_data
    df = generate_synthetic_data(400)
    print(f'Données synthétiques générées : {df.shape}')

df.head()

In [ ]:
print('=== Informations générales ===')
print(f'Dimensions : {df.shape}')
print(f'\nTypes de données :')
print(df.dtypes.value_counts())
print(f'\nValeurs manquantes : {df.isnull().sum().sum()}')
df.describe()

In [ ]:
# Distribution de la variable cible G3 (note finale)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['G3'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution de la note finale (G3)')
axes[0].set_xlabel('Note (0-20)')
axes[0].set_ylabel('Nombre d\'étudiants')
axes[0].axvline(x=10, color='red', linestyle='--', label='Seuil de réussite (10)')
axes[0].legend()

pass_rate = (df['G3'] >= 10).mean() * 100
axes[1].pie([pass_rate, 100 - pass_rate],
            labels=['Réussite (G3≥10)', 'Échec (G3<10)'],
            colors=['#2ecc71', '#e74c3c'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title(f'Taux de réussite : {pass_rate:.1f}%')

plt.tight_layout()
plt.show()

In [ ]:
# Corrélation entre variables numériques
num_cols = df.select_dtypes(include=np.number).columns
corr = df[num_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Matrice de corrélation — Features numériques')
plt.tight_layout()
plt.show()

In [ ]:
# Analyse des features les plus corrélées avec G3
g3_corr = corr['G3'].drop('G3').sort_values(ascending=False)
plt.figure(figsize=(10, 6))
g3_corr.plot(kind='bar', color=['#2ecc71' if v > 0 else '#e74c3c' for v in g3_corr])
plt.title('Corrélation des features avec G3 (note finale)')
plt.xlabel('Features')
plt.ylabel('Corrélation de Pearson')
plt.xticks(rotation=45)
plt.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Analyse des variables catégorielles
cat_cols = df.select_dtypes(include='object').columns
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols[:8]):
    df_grouped = df.groupby(col)['G3'].mean().sort_values()
    df_grouped.plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(f'Moyenne G3 par {col}')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].axhline(y=10, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Influence des variables catégorielles sur G3', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Préparation des Données (Data Preparation)

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

CAT_COLS = [
    'school', 'sex', 'address', 'famsize', 'Pstatus',
    'Mjob', 'Fjob', 'reason', 'guardian',
    'schoolsup', 'famsup', 'paid', 'activities',
    'nursery', 'higher', 'internet', 'romantic'
]

df_proc = df.copy()

# 1. Encodage des variables catégorielles
encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))
    encoders[col] = le

# 2. Variable cible binaire : réussite (1) ou échec (0)
df_proc['pass'] = (df_proc['G3'] >= 10).astype(int)
print(f'Répartition classes : {df_proc["pass"].value_counts().to_dict()}')

# 3. Séparation features / target
feature_cols = [c for c in df_proc.columns if c not in ['G3', 'pass']]
X = df_proc[feature_cols].values
y = df_proc['pass'].values

# 4. Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nFeatures utilisées ({len(feature_cols)}) :')
print(feature_cols)

In [ ]:
# Visualisation de la distribution avant / après normalisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Avant normalisation — quelques features numériques
num_feats = ['age', 'absences', 'G1', 'G2', 'studytime']
df_proc[num_feats].boxplot(ax=axes[0])
axes[0].set_title('Avant normalisation')
axes[0].tick_params(axis='x', rotation=45)

# Après normalisation
idx = [feature_cols.index(f) for f in num_feats if f in feature_cols]
pd.DataFrame(X_scaled[:, idx], columns=num_feats).boxplot(ax=axes[1])
axes[1].set_title('Après normalisation (StandardScaler)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 4. Réduction de Dimensionnalité — ACP (PCA)

In [ ]:
from sklearn.decomposition import PCA

# PCA avec toutes les composantes pour analyser la variance
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

# Variance expliquée cumulée
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = np.argmax(cumvar >= 0.95) + 1

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
        pca_full.explained_variance_ratio_, color='steelblue')
plt.xlabel('Composante principale')
plt.ylabel('Variance expliquée')
plt.title('Variance expliquée par composante')

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumvar) + 1), cumvar, 'b-o', markersize=4)
plt.axhline(y=0.95, color='red', linestyle='--', label='Seuil 95%')
plt.axvline(x=n_95, color='green', linestyle='--', label=f'{n_95} composantes')
plt.xlabel('Nombre de composantes')
plt.ylabel('Variance expliquée cumulée')
plt.title('Variance cumulée — Choix du nombre de composantes')
plt.legend()

plt.tight_layout()
plt.show()
print(f'Composantes nécessaires pour 95% de variance : {n_95}')

In [ ]:
# Application de la PCA avec 95% de variance
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Dimensions originales : {X_scaled.shape}')
print(f'Dimensions après PCA  : {X_pca.shape}')
print(f'Réduction : {X_scaled.shape[1]} → {X_pca.shape[1]} composantes')
print(f'Variance expliquée totale : {sum(pca.explained_variance_ratio_):.3f}')

# Visualisation en 2D (PC1 vs PC2)
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y,
                      cmap='RdYlGn', alpha=0.7, edgecolors='white', linewidths=0.5)
plt.colorbar(scatter, label='Réussite (1) / Échec (0)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('Projection ACP — Espace réduit (PC1 vs PC2)')
plt.tight_layout()
plt.show()

---
## 5. Apprentissage Non Supervisé — K-Means Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Méthode Elbow — choisir le bon k
inertias = []
sil_scores = []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_pca, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'b-o')
axes[0].set_xlabel('Nombre de clusters k')
axes[0].set_ylabel('Inertie')
axes[0].set_title('Méthode Elbow')
axes[0].axvline(x=3, color='red', linestyle='--', label='k=3 (choix optimal)')
axes[0].legend()

axes[1].plot(K_range, sil_scores, 'g-o')
axes[1].set_xlabel('Nombre de clusters k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Score Silhouette par k')
axes[1].axvline(x=3, color='red', linestyle='--', label='k=3')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# K-Means avec k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_pca)

sil = silhouette_score(X_pca, clusters)
print(f'Silhouette Score (k=3) : {sil:.4f}')

# Visualisation des clusters
cluster_names = {0: 'At-Risk', 1: 'Average', 2: 'High-Performing'}
colors = ['#e74c3c', '#f39c12', '#2ecc71']

plt.figure(figsize=(10, 6))
for c in range(3):
    mask = clusters == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                label=f'Cluster {c} — {cluster_names[c]} ({mask.sum()})',
                color=colors[c], alpha=0.7, edgecolors='white', linewidths=0.5)

centers_pca = kmeans.cluster_centers_
plt.scatter(centers_pca[:, 0], centers_pca[:, 1],
            marker='X', s=200, c='black', label='Centroïdes', zorder=5)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title(f'K-Means Clustering (k=3) — Silhouette: {sil:.3f}')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Analyse des clusters : caractéristiques moyennes
df_cluster = df_proc.copy()
df_cluster['cluster'] = clusters
df_cluster['cluster_name'] = df_cluster['cluster'].map(cluster_names)

cluster_profile = df_cluster.groupby('cluster_name')[['G1', 'G2', 'G3', 'studytime', 'failures', 'absences']].mean().round(2)
print('Profil moyen par cluster :')
cluster_profile

---
## 6. Apprentissage Supervisé — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix, RocCurveDisplay)

X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

acc     = accuracy_score(y_test, y_pred)
f1      = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f'Accuracy  : {acc:.4f}')
print(f'F1-Score  : {f1:.4f}')
print(f'ROC-AUC   : {roc_auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Échec', 'Réussite']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Prédit Échec', 'Prédit Réussite'],
            yticklabels=['Réel Échec', 'Réel Réussite'])
axes[0].set_title('Matrice de Confusion')

# Courbe ROC
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1],
                                  name=f'Random Forest (AUC={roc_auc:.3f})',
                                  color='steelblue')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('Courbe ROC')

plt.tight_layout()
plt.show()

---
## 7. Conclusion et Tableau d'Évaluation Final

In [ ]:
summary = pd.DataFrame([
    {
        'BO': 'Réduire la dimensionnalité des données étudiantes',
        'DSO': 'Compression de l\'espace de features',
        'Algorithme': 'ACP (PCA)',
        'Métrique': 'Variance expliquée',
        'Résultat': f'{sum(pca.explained_variance_ratio_):.1%} avec {pca.n_components_} composantes',
        'Conclusion': 'Réduction significative de dimensionnalité sans perte d\'information'
    },
    {
        'BO': 'Segmenter les étudiants par profil académique',
        'DSO': 'Clustering non supervisé',
        'Algorithme': 'K-Means (k=3)',
        'Métrique': 'Silhouette Score',
        'Résultat': f'{sil:.4f}',
        'Conclusion': '3 profils distincts : At-Risk, Average, High-Performing'
    },
    {
        'BO': 'Prédire la réussite à l\'examen de rattrapage',
        'DSO': 'Classification supervisée réussite/échec',
        'Algorithme': 'Random Forest (100 arbres)',
        'Métrique': 'Accuracy / F1 / ROC-AUC',
        'Résultat': f'Acc={acc:.3f}, F1={f1:.3f}, AUC={roc_auc:.3f}',
        'Conclusion': 'Modèle performant pour identifier les étudiants à risque avant le rattrapage'
    }
])

summary

### Lien avec l'ODD 4 — Éducation de Qualité

Ce projet ML contribue concrètement à l'ODD 4 en :
1. **Détection précoce** : Le modèle Random Forest identifie les étudiants à risque avant les examens, permettant une intervention pédagogique ciblée
2. **Équité** : En analysant tous les profils (via K-Means), le système s'assure qu'aucun groupe d'étudiants n'est oublié
3. **Efficacité des ressources** : Les enseignants peuvent concentrer leur soutien sur les étudiants du cluster 'At-Risk'
4. **Intégration plateforme** : Le modèle est exposé via une API REST (FastAPI) intégrée dans la plateforme éducative, rendant les prédictions accessibles en temps réel